In [ ]:
# Install the required libraries
!pip install -qU langgraph langchain langchain-openai faiss-cpu langchain_community

**Importing Packages**

In [ ]:
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_openai import ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, List, Literal, Optional
from openai import OpenAI
import os, json, mimetypes, base64
import faiss
from langchain_openai import OpenAIEmbeddings
import pickle
import uuid
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
import json
import os
from getpass import getpass

OPEN_AI_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

**Managing Directories Paths**

In [ ]:
# Set all path variables
PROJECT_PATH = Path("/content/capstone_project/")
PROCESSED_DIR = PROJECT_PATH / "data" / "processed"
IMAGE_DIR = PROJECT_PATH / "data" / "images"
POLICY_FAISS_INDEX_PATH = PROJECT_PATH / "outputs" / "vector_stores" / "policy_faiss_index"
PRODUCT_FAISS_INDEX_PATH = PROJECT_PATH / "outputs" / "vector_stores" / "product_faiss_index"
OUTPUT_LOG_DIR = PROJECT_PATH / "outputs" / "logs"
MODEL_PKL_PATH = PROJECT_PATH / "outputs" / "models"
OUTPUT_JSON_FILE = PROJECT_PATH / "outputs" / "logs" / "assistant_agent_evaluation.jsonl"

In [ ]:
# Prompt for the OpenAI API Key securely
os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API Key: ")

# Optional: required only for institutional/proxy endpoints
base_url = getpass(
    "Enter OPENAI_BASE_URL (press Enter if not required): "
).strip()

if base_url:
    os.environ["OPENAI_BASE_URL"] = base_url
else:
    os.environ.pop("OPENAI_BASE_URL", None)


**Loading of previously stored Indexes**

In [ ]:
# Instantiate your embedding model (it must be the EXACT same model used to save it)
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL)

1) Loading Policy FAISS index

In [ ]:
vector_store = FAISS.load_local(
    folder_path=POLICY_FAISS_INDEX_PATH,
    embeddings=embeddings,
    allow_dangerous_deserialization=True  # Required to safely unpack the local .pkl file
)

print("Vector store loaded successfully.")

# Convert it to a retriever for your RAG pipeline
policy_retriever = vector_store.as_retriever()

**Testing the Policy Retriever independently**

In [ ]:
retrieved_context = policy_retriever.invoke("Can I return shoes after 30 days?")
print(retrieved_context[0].page_content)

2) Loading Product FAISS index

In [ ]:
# Defining the pths where indexes and pkl files are stored
file_path = os.path.join(PRODUCT_FAISS_INDEX_PATH, "index.faiss")
docs_path = os.path.join(PRODUCT_FAISS_INDEX_PATH, "docs.pkl")

# Load the native FAISS index first
native_index = faiss.read_index(file_path)

# Load our text documents/chunks
with open(docs_path, "rb") as f:
    docs = pickle.load(f)

## When we instantiate FAISS(...) manually using a low-level native index, LangChain expects a fully configured InMemoryDocstore object right from the start.

# Step - 1. Generate unique matching IDs for each text document chunk
uuids = [str(uuid.uuid4()) for _ in range(len(docs))]

# Step - 2. Build the structural index dictionary mapping (index position -> UUID)
index_to_docstore_id = {i: uuids[i] for i in range(len(docs))}

# Step - 3. Create a valid structural InMemoryDocstore containing the documents
docstore = InMemoryDocstore({uuids[i]: docs[i] for i in range(len(docs))})

# Wrap the native index into LangChain's FAISS class
# This bridges the gap between low-level faiss and LangChain tools
vector_store = FAISS(
    embedding_function=embeddings,
    index=native_index,
    docstore=docstore,
    index_to_docstore_id=index_to_docstore_id
)

# Manually register your loaded text documents into the new vector store wrapper
vector_store.add_documents(docs)

# 6. Create the retriever
product_retriever = vector_store.as_retriever()
print("Retriever successfully configured!")

**Testing the Product Retriever independently**

In [ ]:
retriver = product_retriever.invoke("recommend good running shoes under 1000")
print(retriver[0].page_content)

3) Loading Prediction Model

In [ ]:
# Load the bundle
prediction_model = MODEL_PKL_PATH / "prediction_model.pkl"
bundle = joblib.load(prediction_model)

loaded_prediction_model = bundle["model"]
raw_feature_names_loaded = bundle["raw_feature_names"]
transformed_feature_names_loaded = bundle["transformed_feature_names"]
target_name_loaded = bundle["target_name"]
class_names_loaded = bundle["class_names"]

# Load the processed dataset
processed_csv = PROCESSED_DIR / "retail_processed.csv"
retail_processed_df = pd.read_csv(processed_csv)

In [ ]:
retail_processed_df.head()

In [ ]:
# The 'retail_processed.csv' contains the base/original columns and the Feature-engineered columns wuld be missing.
# We need to build and update the dataset with those newly created columns

def build_engineered_features(df):
  retail_df = df.copy()

  retail_df['discounted_amt'] = (retail_df['base_price'] - retail_df['final_price'])
  retail_df['effective_customer_order_value'] = (retail_df['avg_order_value'] * retail_df['total_orders'])
  retail_df["customer_segment"] = (
    retail_df["income_band"].astype(str)
    + "_"
    + retail_df["membership_level"].astype(str)
  )

  bins = [-1, 30, 90, 150]
  labels = ['Active', 'At-Risk', 'In-Active']

  retail_df['customer_state'] = pd.cut(retail_df['days_since_last_purchase'], bins=bins, labels=labels)

  return retail_df

4) Loading Customer Segmentation Model

In [ ]:
# Load the bundle
segmentation_model = MODEL_PKL_PATH / "customer_segmentation_model.pkl"
bundle = joblib.load(segmentation_model)

prediction_model = bundle["model"]
scaler = bundle["scaler"]
cluster_features = bundle["feature_names"]
cluster_profiles = bundle["cluster_profiles"]
cluster_namings = bundle["cluster_names"]
cluster_description = bundle["cluster_description"]
segment_analysis = bundle["segment_analysis"]

print("Model:", prediction_model)
print("\n Features:", cluster_features)
print("\n Loaded Cluster Profile:")
display(cluster_profiles)
print("\n Cluster Namings:")
display(cluster_namings)
print("\n Loaded Segment Analysis:")
display(segment_analysis)
print("\n Number of clusters:", prediction_model.n_clusters)

In [ ]:
# Now, consider a scenario when a user ask a query not to identify a cluster to any specific Customer, for e.g. "Which customer's segment is highly engaged ?"
# To support such scenarios, we will not be able to predict via the Loaded model and answer to the user.

# Now, we have loaded details of 'cluster profiles' and 'segment_analysis' but we may not ask LLM to identify the Segment directly for us.
# Instead, we may require to handle this use-case manually and then maybe we may ask LLM to frame the user-friendly response for us.

# To support this, we will create a metric mappings rules for different intents ( metrics ) for which user has asked a query and
# will try to use Pandas to identify the best cluster from the 'segment_analysis' data. ( We will ask LLM to identify the intents out of user's query)

metric_mapping_rules = {
    "price_sensitivity": {
        "metric" : "avg_discount_percent",
        "direction" : "max"
    },
    "frequency": {
        "metric" : "total_orders",
        "direction" : "max"
    },
    "highest_value": {
        "metric": "avg_order_value",
        "direction": "max"
    },
    "low_recency" : {
        "metric" : "days_since_last_purchase",
        "direction" : "max"
    },
    "highly_engaged" : {
        "metric" : "avg_time_on_page",
        "direction" : "max"
    },
    "cart": {
        "metric": "cart_rate",
        "direction": "max"
    },
}

**Testing the Customer Segmentation independently**

In [ ]:
# Calculate first the Dataset with one row per customer
df_customers = retail_processed_df.groupby("customer_id").agg(
    total_orders=("total_orders", "first"),
    avg_order_value=(
        "avg_order_value", "first"
    ),
    days_since_last_purchase=(
        "days_since_last_purchase", "first"
    ),
    avg_time_on_page=("time_on_page", "mean")
).reset_index()

In [ ]:
# Take few customer-Ids from the dataset whose cluster's was already assigned during training.
test_customer_ids = ["C105", "C205", "C255", "C349"]

for customer_id in test_customer_ids:
    customer = df_customers[
        df_customers["customer_id"] == customer_id
    ].copy()

    if customer.empty:
        print(customer_id, "→ Customer not found")
        continue

    X_customer = customer[cluster_features]
    X_scaled = scaler.transform(X_customer)

    cluster = int(prediction_model.predict(X_scaled)[0])
    print(f"{customer_id} belongs to Cluster {cluster}")

Creating the LLM Prompts

In [ ]:
PRODUCT_RAG_SYSTEM_PROMPT = """
You are a Retail Product Intelligence component of an Intelligent Retail Decision Assistant.

Your job is to answer product recommendation and comparison requests using ONLY
the retrieved product context provided to you.

STRICT RELIABILITY RULES:

1. Recommend or compare ONLY products that appear in the retrieved context.

2. Do not invent product names, product IDs, prices, discounts, ratings,
   categories, sizes, colors, availability, or any other attributes.

3. If the retrieved context does not contain enough information to answer the
   user's request, clearly state this in the response.

4. Do not use outside knowledge.

5. When explaining why a product is recommended, use only information explicitly
   available in the retrieved context.

6. If the user specifies constraints such as price, category, style, occasion,
   rating, or other requirements, check those constraints against the retrieved
   context.

7. Do not claim that a product satisfies a requirement unless the retrieved
   context provides evidence for that claim.

8. Keep the response concise and helpful.

Return the answer strictly according to the provided structured output schema.
"""

In [ ]:
IMAGE_EXTRACTION_QUERY_PROMPT = """
Given an image of a product, extract a retrieval query.

Return STRICT JSON with keys:
- category_guess mut be exacty one of : "shoes","phones","laptops","smartwatches","unknown"
- brand_guess: string or null
- key_attributes: list of short phrases
- search_query: short semantic search query (category + key hints)

Rules:
- Avoid hallucinating specs.
- Be conservative.
"""

In [ ]:
PREDICTION_RESPONSE_PROMPT = """
You are an Intelligent Retail Decision Assistant responsible for presenting the result of a machine-learning purchase prediction.

The purchase prediction has already been generated by a trained machine-learning model. You MUST NOT change, reinterpret, or recalculate the model's prediction or probability.

Your job is only to convert the supplied prediction result into a clear, concise, customer-friendly response.

Rules:

 1. If the prediction is "Purchased", say that the model predicts the customer is likely to purchase the product.
 2. If the prediction is "Not Purchased", say that the model predicts the customer is unlikely to purchase the product.
 3. Always mention the purchase probability as a percentage.
 4. Do not say "You are predicted to..." because the prediction is made by the model, not by the assistant.
 5. Do not perform your own prediction.
 6. Do not present the prediction as certain.
 7. Keep the response to one or two sentences.
 8. Return plain text only.

Input:

Prediction Result: {prediction_result}
"""

In [ ]:
POLICY_QA_SYSTEM_PROMPT = """
You are the Policy Intelligence component of an Intelligent Retail Decision Assistant.

Your job is to answer questions about retail policies using ONLY the retrieved
policy context.

STRICT RELIABILITY RULES:

1. Use ONLY the retrieved policy context.

2. Do not use outside knowledge or general assumptions about retail policies.

3. Every policy answer must include at least one doc_id.

4. If a requested doc_id does not exist, do not silently substitute another document.

5. Include a short verbatim quote for any policy rule.

6. Do not invent policy rules, conditions, exceptions, time periods, fees,
   eligibility criteria, or procedures.

7. If the answer is not explicitly supported by the retrieved policy context,
   set insufficient_information to true.

8. Clearly distinguish between:
   - information explicitly stated in the policy
   - information that is not available in the retrieved context

9. Include relevant conditions, restrictions, and exceptions when they are
   explicitly present in the retrieved policy context.

10. Do not claim that a customer or product is eligible unless the retrieved
   policy information supports that conclusion.

11. Keep the answer clear, concise, and faithful to the retrieved policy context.

Return the answer strictly according to the provided structured output schema.
"""

In [ ]:
ROUTER_PROMPT = """
You are the Router Classification component of an Intelligent Retail Decision Assistant.

Your job is to classify the user's query into exactly ONE of the following categories:

1. recommendation
   Return this category when the user is asking for:
   - product recommendations
   - suitable products based on preferences or requirements
   - product search
   - product comparison
   - product features, prices, ratings, categories, or other catalog information

2. prediction
   Return this category when the user is asking for:
   - purchase likelihood
   - probability of purchase
   - whether a customer is likely to purchase
   - a prediction based on customer or product-related data

   IMPORTANT:
   Do NOT classify general customer analysis or customer segment
   questions as prediction

3. policy
   Return this category when the user is asking about:
   - return or refund rules
   - exchanges
   - shipping policies
   - warranties

4. segmentation
   Queries asking about customer segments, clusters, customer groups,
   behavioural profiles, or comparisons/analysis of customer groups.

   This includes:
   - identifying which segment a specific customer belongs to
   - describing or comparing customer segments
   - identifying the most/least active segment
   - identifying high-value or low-value segments
   - analyzing price-sensitive customer segments

Choose the category that represents the PRIMARY intent of the user's query.

Do not answer the user's question.
Do not retrieve any information.
Only classify the query.

Return only one category:
recommendation
prediction
policy
segmentation

Return the result strictly according to the structured output schema.
"""

In [ ]:
CUSTOMER_DETAILS_EXTRACT_PROMPT = """
Extract the customer ID and product ID from the user's query.
Do not predict anything.
"""

In [ ]:
SEGMENTATION_INTENT_PROMPT = """
You are the Customer Segmentation Intent Classification component of an Intelligent Retail Decision Assistant.

Your job is to identify the user's intended customer-segmentation analysis from the query.

Return ONLY one of the allowed segmentation intents. Do not calculate any values, identify the cluster, or generate an explanation

Identify from the give intents :

1. frequency — User asks which segment has the highest purchase/order frequency.
   Examples: "Which segment buys most frequently?", "Which customer group has the most orders?"

2. price_sensitivity - User asks which segment appears most sensitive to discounts or price-related promotions.
   Use this intent for queries about price sensitivity, discount-seeking behaviour, bargain-seeking behaviour, or dependence on discounts.
   Examples: "Which segment is most price-sensitive?", "Which customer group responds most to discounts?"

3. low_recency — User asks which segment is least recently active, most inactive, or most at risk due to longer time since last purchase.
   Examples: "Which segment is most inactive?", "Which customer group is at risk of churning?"

4. highest_value - User asks which segment has the highest spending or average order value.
   Examples: "Which segment spends the most?", "Which customer group has the highest average order value?"

5. cart - User asks which segment has the highest add-to-cart behaviour.
   Examples: "Which segment adds products to cart most often?"

6. highly_engaged - User asks which segment has the highest browsing/engagement behaviour.
   Examples: "Which segment is most engaged?", "Which customer group spends the most time browsing?"

7. segment_overview — User asks for a general description, summary, or characteristics of the customer segments without asking for any specific metric.
   Examples: "Describe the customer segments.", "What are the different customer groups?"

Following rules must be followed :

1) Select exactly ONE intent.
2) If multiple metrics are mentioned, select the metric that is the main focus of the question.
"""

In [ ]:
SEGMENTATION_RESPONSE_PROMPT = """
You are the Customer Segmentation Response Generation component of an Intelligent Retail Decision Assistant.

Generate a concise, user-friendly answer to the user's query using ONLY the provided segmentation results.

Conisder following while generating the response:

1) Do not recalculate metrics or invent information.
2) Clearly mention the relevant segment, segment name, metric/value, and briefly explain why it supports the answers.
3) Briefly explain what the metric means in the context of the user's question.
4) Keep the response concise and user friendy.
5) Do not use overly technical terminology unless the user explicitly asks for technical details.
6) If the supplied data is insufficient to answer the question, state that clearly instead of making an assumption.

Following are the inputs provided:
User Query:
{user_query}

Identified Segment:
{segment_name}

Identified Metric:
{metric}

Cluster Profiles:
{cluster_profiles}

Segment Analysis:
{segment_analysis}

Segmentation Summary:
{segmentation_summary}
"""

Integrating the Pipeline through Langgraph

1) Creating the AgentAssistantState

In [ ]:
class RetailAssistantState(TypedDict):
  user_query: str
  image_path: str | None
  customer_id: str
  product_id: str
  purchase_probability: float
  prediction_label: str
  query_type: str
  final_response: str

2) Defining Product RAG Structured Schema

In [ ]:
class RecommendedProduct(BaseModel):
    product_id: str = Field(description = "Product ID exactly as provided in the retrieved context.")
    product_name: str = Field(description = "Product name exactly as provided in the retrieved context.")
    reason: str = Field(description = "Brief reason for recommending the product, based only on retrieved information.")
    citations: List[str] = Field(default_factory=list)

class ProductCatalogOutput(BaseModel):
    query_type: Literal["recommendation","comparison"]
    user_need: str = Field(description = "A concise summary of what the user is looking for.")
    recommended_products: List[RecommendedProduct]
    response: str = Field(description = "A concise user-facing response based only on retrieved product context.")


3) Defining the Router Classification Schema

In [ ]:
class RouterOutput(BaseModel):
    query_type: Literal["recommendation","prediction","segmentation","policy"] = Field(description = "The primary intent required to route the user's query to the appropriate node.")
    classification_reasoning: str = Field(description = "Brief explanation of why this route was selected.")

4) Defining Policy QA Structured Schema

In [ ]:
class PolicySource(BaseModel):
  policy_id: str = Field(description = "Policy document ID from the retrieved context.")
  policy_details: str = Field(description = "Brief description of the policy information relevant to the answer.")

class PolicyQAOutput(BaseModel):
  user_need: str = Field(description = "A concise summary of what the user is looking for.")
  relevant_response: str = Field(description = "The relevant response based only on the retrieved policy context.")
  policy_source: List[PolicySource] = Field(description ="Relevant policy documents used to support the answer.")

5) Defining Customer Details Extraction Request Schema

In [ ]:
class CustomerDetailsExtractionRequest(BaseModel):
    customer_id: str | None
    product_id: str | None

6. Defining Customer Segmentation Intent Classification Schema

In [ ]:
class SegmentIntentClassification(BaseModel):
  intent_type: Literal["frequency","price_sensitivity","low_recency","highest_value","cart","highly_engaged","segment_overview"] = Field(description = "The primary intent required to classify the segmentatin intent or route based on the user's query.")
  classification_reasoning: str = Field(description = "Brief explanation of why this route was selected.")

7. Defining Customer Segmentation Response Schema

In [ ]:
class SegmentationResponse(BaseModel):
  response: str = Field(description="The generated reponse out of the segment analysis based on the identified intent")

8. Defining Schema for Multimodal RAG

In [ ]:
class ImageQueryExtraction(BaseModel):
    category_guess: Optional[Literal["shoes","phones","laptops","smartwatches","unknown"]] = "unknown"
    brand_guess: Optional[str] = None
    key_attributes: List[str] = Field(default_factory=list)
    search_query: str

9.  Instantiating LLM And their Structured output objects

In [ ]:
retail_assistant_llm = ChatOpenAI(
    model=OPEN_AI_MODEL,
    temperature=0
  )

# Creating the structured output LLM objects for different nodes
structured_product_llm = retail_assistant_llm.with_structured_output(ProductCatalogOutput)
structured_router_llm = retail_assistant_llm.with_structured_output(RouterOutput)
structured_policy_llm = retail_assistant_llm.with_structured_output(PolicyQAOutput)
structured_cust_extract_llm = retail_assistant_llm.with_structured_output(CustomerDetailsExtractionRequest)
structured_seg_intent_cls_llm = retail_assistant_llm.with_structured_output(SegmentIntentClassification)
structured_seg_res = retail_assistant_llm.with_structured_output(SegmentationResponse)

10. Observability

In [ ]:
def log_event(event,route,query,query_type,final_response,customer_id=None,product_id=None,probability=None,prediction=None,policy_id=None):
  output_data = {
      "event":event,
      "route":route,
      "query":query,
      "query_type":query_type,
  }
  if customer_id is not None:
    output_data["customer_id"] = customer_id

  if product_id is not None:
    output_data["product_id"] = product_id

  if probability is not None:
    output_data["probability"] = float(probability)

  if prediction is not None:
    output_data["prediction"] = prediction

  if policy_id is not None:
    output_data["policy_id"] = policy_id

  output_data["final_response"] = final_response

  with open(OUTPUT_JSON_FILE, "a", encoding="utf-8") as outfile:
    outfile.write(json.dumps(output_data, ensure_ascii=False) + "\n")

11. Defining the Graph Node Functions

In [ ]:
def image_to_data_url(image_path: str) -> str:
    mime_type, _ = mimetypes.guess_type(image_path)
    if mime_type is None:
        mime_type = "image/png"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode("utf-8")
    return f"data:{mime_type};base64,{b64}"

def build_image_retrieval_query(image_extraction: Optional[ImageQueryExtraction]) -> str:
  parts = []
  parts.append(image_extraction.search_query.strip())
  if image_extraction.brand_guess:
      parts.append(f"brand {image_extraction.brand_guess}")
  if image_extraction.key_attributes:
      parts.append(" ".join(image_extraction.key_attributes))

  return " | ".join(parts) if parts else ""

def extract_query(image_path: str) -> ImageQueryExtraction:
    img_url = image_to_data_url(image_path)

    messages = [
    SystemMessage(
        content="Return valid JSON only. No markdown."
    ),
    HumanMessage(
        content=[
            {
                "type": "text",
                "text": IMAGE_EXTRACTION_QUERY_PROMPT
            },
            {
                "type": "image_url",
                "image_url": {"url": img_url }
            }
        ]
      )
    ]

    response = retail_assistant_llm.invoke(messages)
    raw = response.content
    data = json.loads(raw)
    return ImageQueryExtraction(**data)

def product_RAG_node(state: RetailAssistantState):
  # Fetch the User' query from the state object
  user_query = state['user_query']
  img_path = state['image_path']

  if img_path is not None:
    image_extraction = extract_query(img_path)
    retrieval_query = build_image_retrieval_query(image_extraction)
    merged_retrieval_query = f"{retrieval_query} {user_query}"
    retrieved_context = product_retriever.invoke(merged_retrieval_query)
    user_query = merged_retrieval_query
  else:
    retrieved_context = product_retriever.invoke(user_query)

  # Invoking the LLM to get the structured response for product recommendation
  user_message = (
    f"User Query: {user_query}\n"
    f"Retrieved Context: {retrieved_context}\n"
  )
  messages = [
    SystemMessage(content=PRODUCT_RAG_SYSTEM_PROMPT),
    HumanMessage(content=user_message)
  ]

  response = structured_product_llm.invoke(messages)
  log_event(event="recommendation",route="Product RAG",query=user_query,query_type=state["query_type"],final_response=response.response)
  return {'final_response': response}

def policy_qa_node(state: RetailAssistantState):
  # Fetch the User' query from the state object
  user_query = state['user_query']
  # Fetching the Retrieved context from the Policy Index
  retrieved_context = policy_retriever.invoke(user_query)
  # Invoking the LLM to get the structured response for policy information
  user_message = (
    f"User Query: {user_query}\n"
    f"Retrieved Context: {retrieved_context}\n"
  )
  messages = [
    SystemMessage(content=POLICY_QA_SYSTEM_PROMPT),
    HumanMessage(content=user_message)
  ]

  response = structured_policy_llm.invoke(messages)
  log_event(event="policy",route="Policy RAG",query=user_query,query_type=state["query_type"],policy_id=response.policy_source[0].policy_id,final_response=response.relevant_response)
  return {'final_response': response}

def purchase_prediction_node(state: RetailAssistantState):
  # Fetch the User' query from the state object
  user_query = state['user_query']

  # From the user's query, we would require to extract the Customer-Id and Product-Id.
  # Invoking LLM to extract those details
  user_message = (
    f"User Query: {user_query}"
  )
  messages = [
    SystemMessage(content=CUSTOMER_DETAILS_EXTRACT_PROMPT),
    HumanMessage(content=user_message)
  ]
  extraction_response = structured_cust_extract_llm.invoke(messages)

  customer_id = extraction_response.customer_id
  product_id = extraction_response.product_id

  # Once we receives the Customer-Id and Product-Id, we will now fetch the respective records from the loaded dataset
  if customer_id is not None and product_id is not None:
    customer_matching_record = retail_processed_df[(retail_processed_df['customer_id'] == customer_id) & (retail_processed_df['product_id'] == product_id)]
  elif customer_id is not None:
    customer_matching_record = retail_processed_df[(retail_processed_df['customer_id'] == customer_id)]
  elif product_id is not None:
    customer_matching_record = retail_processed_df[(retail_processed_df['product_id'] == product_id)]
  # Handling the case when user provides a query withpout mentioning the customer-id and product-id. We should return from here.
  else:
    return {'final_response' : "Please provide a customer ID or product ID to perform the purchase prediction."}

  # Update the Matching-Record and enrich those with the Feature engineered columns
  customer_matching_record = build_engineered_features(customer_matching_record)
  # Above matchinh records contains data from all the columns, we would be requiring to know the values from the final features being used when we trained the model
  final_input_df = customer_matching_record[raw_feature_names_loaded].copy()

  # Lets now perform the probability, make the Prediction and set the appropriate label
  purchase_probability = loaded_prediction_model.predict_proba(final_input_df)[0][1]
  purchase_prediction = int(purchase_probability >= 0.5)
  prediction_label = class_names_loaded[purchase_prediction]

  prediction_result = {
          "prediction" : prediction_label,
          "purchase_probabiity" : purchase_probability
      }

  user_message = (
    f"Prediction Result: {prediction_result}\n"
    )
  messages = [
    SystemMessage(content=PREDICTION_RESPONSE_PROMPT),
    HumanMessage(content=user_message)
  ]
  final_response = retail_assistant_llm.invoke(messages)
  log_event(event="prediction",
            route="Classical ML (Prediction Probability)",
            query=user_query,
            query_type=state["query_type"],
            customer_id=customer_id,
            product_id=product_id,
            probability=np.float32(purchase_probability),
            prediction=prediction_label,
            final_response=final_response.content)

  return {'customer_id': customer_id,'product_id': product_id,'purchase_probability': purchase_probability,'prediction_label': prediction_label,'final_response': final_response.content}

def customer_segmentation_node(state: RetailAssistantState):
  # Fetch the User' query from the state object
  user_query = state['user_query']

  # From the user's query, we would require to extract the Customer-Id
  # Invoking LLM to extract those details
  user_message = (
    f"User Query: {user_query}"
  )
  messages = [
    SystemMessage(content=CUSTOMER_DETAILS_EXTRACT_PROMPT),
    HumanMessage(content=user_message)
  ]
  extraction_response = structured_cust_extract_llm.invoke(messages)

  customer_id = extraction_response.customer_id

  if customer_id is not None:
    # We have received the customer-id, loading the details based on it and make the prediction
    customer = df_customers[
        df_customers["customer_id"] == customer_id
    ].copy()

    X_customer = customer[cluster_features]
    X_scaled = scaler.transform(X_customer)

    cluster = int(prediction_model.predict(X_scaled)[0])
    # Now, we have got the cluster, lets get its details
    segment_name = cluster_namings[cluster]
    segment_desc = cluster_description[cluster]

     # Generate the overall segmentation result out of the above details
    segmentation_summary = {
        "segment_name" : segment_name,
        "segment_description" : segment_desc,
        "cluster_value" : cluster
    }
  else:
    # If we haven't received customer-id, we may need to chek if user tried to ask query specific to any segment
    # lets invoke the LLM to understand the segmentation intent
    user_message = (
    f"User Query: {user_query}\n"
    )
    messages = [
      SystemMessage(content=SEGMENTATION_INTENT_PROMPT),
      HumanMessage(content=user_message)
    ]

    intent_response = structured_seg_intent_cls_llm.invoke(messages)
    intent = intent_response.intent_type

    if intent_response.intent_type != "":
      metric = metric_mapping_rules[intent]['metric']
      # We have the dataFrame of 'segment_analysis' and 'cluster_profiles' with mean data of different metrics for all the clusters
      # Let's find the maximum out of it for the identified metric
      if metric in cluster_features:
        cluster = cluster_profiles[metric].idxmax()
        segment_details = cluster_profiles[metric].max()
        segment_details = f"{segment_details:.2f}"
      else:
        cluster = segment_analysis[metric].idxmax()
        segment_details = segment_analysis[metric].max()
        segment_details = f"{segment_details:.2f}"

      # Now, we have got the cluster, lets get its details
      segment_name = cluster_namings[cluster]
      segment_desc = cluster_description[cluster]


      # Generate the overall segmentation result out of the above details
      segmentation_summary = {
          "metric_name" : metric,
          "segment_name" : segment_name,
          "segment_description" : segment_desc,
          "cluster_value" : cluster
      }

  user_message = (
    f"Segmentation Summary: {segmentation_summary}\n"
    )
  messages = [
    SystemMessage(content=SEGMENTATION_RESPONSE_PROMPT),
    HumanMessage(content=user_message)
  ]
  final_response = structured_seg_res.invoke(messages)
  log_event(event="segmentation",
            route="Classical ML (Customer Segmentation)",
            query=user_query,
            query_type=state["query_type"],
            customer_id=customer_id,
            final_response=final_response.response)
  return {'final_response': final_response}


def retail_assistant_router(state: RetailAssistantState):
  # Fetch the User' query from the state object
  user_query = state['user_query']
  # Invoking the LLM to make the classification of the query type for the user's query
  user_message = (
    f"User Query: {user_query}"
  )
  messages = [
    SystemMessage(content=ROUTER_PROMPT),
    HumanMessage(content=user_message)
  ]

  response = structured_router_llm.invoke(messages)
  return {'query_type': response.query_type}

def query_type_router(state: RetailAssistantState) -> Literal["product_recommendation", "purchase_prediction","customer_segmentation","policy_assistance"] :
  # Fetch the query type classified for the user's query
  query_type = state['query_type']
  # Return the node type based on the query type
  if query_type == 'recommendation':
    return 'product_recommendation'
  elif query_type == 'prediction':
    return 'purchase_prediction'
  elif query_type == 'segmentation':
    return 'customer_segmentation'
  else:
    return 'policy_assistance'


12) Defining Graphs nodes and edges

In [ ]:
graph = StateGraph(RetailAssistantState)
graph.add_node('classification_router', retail_assistant_router)
graph.add_node('product_recommendation', product_RAG_node)
graph.add_node('purchase_prediction', purchase_prediction_node)
graph.add_node('customer_segmentation', customer_segmentation_node)
graph.add_node('policy_assistance', policy_qa_node)

graph.add_edge(START,'classification_router')
graph.add_conditional_edges('classification_router',query_type_router)
graph.add_edge('product_recommendation', END)
graph.add_edge('purchase_prediction', END)
graph.add_edge('customer_segmentation', END)
graph.add_edge('policy_assistance', END)

retail_assistant_workflow = graph.compile()

**Evaluation Harness cum Manual Tests**

Testing Multimodal cpability of the Assistant

In [ ]:
shoe_img_path = str(IMAGE_DIR/"shoe.jpg")
result = retail_assistant_workflow.invoke({
    "user_query": "Recommend something similar to this image under ₹5,000",
    "image_path": shoe_img_path
})
res = result["final_response"]
print("Assitant Response :: ", res.response)

print("="*20)

watch_img_path = str(IMAGE_DIR/"watch.jpg")
result = retail_assistant_workflow.invoke({
    "user_query": "Suggest a product as shown in this image.",
    "image_path": watch_img_path
})
res = result["final_response"]
print("Assitant Response :: ", res.response)

Testing Texts Queries

In [ ]:
def create_initial_state(query):
  initial_state = {
    'user_query': query,
    'image_path': None,
    'customer_id': None,
    'product_id': None,
    'purchase_probability': None,
    'prediction_label': None,
    'query_type': '',
    'final_response': ''
  }

  return initial_state

In [ ]:
log_file = OUTPUT_LOG_DIR / "assistant_agent_evaluation.txt"
evaluation_queries = [
    "Predict the purchase likelihood for customer C163 and product P1163.",
    "Recommend casual sneakers under ₹4000 for daily wear.",
    "Which customer segment does C102 belong to?",
    "I need running shoes under $200 with a rating above 4",
    "What is the purchase likelihood for customer C142?",
    "Can I return a product after 30 days?",
    "Suggest products for a premium customer who prefers black formal shoes.",
    "Can I exchange an item I purchased?",
    "Which customer segment has the highest purchase frequency?",
    "Which customer segment appears to be the most price-sensitive?",
    "Summarise the top three retrieved products and explain why they fit the query.",
    "Which segment is the most inactive?"
]

with open(log_file, "w", encoding="utf-8") as f:
  for query in evaluation_queries:
    initial_state = create_initial_state(query)
    final_state = retail_assistant_workflow.invoke(initial_state)

    f.write("="*60 + "\n\n")
    f.write(f"User Query:\n{query}\n\n")
    f.write(f"Query Type:\n{final_state['query_type']}\n\n")
    if final_state['query_type'] == 'prediction':
      f.write("Route:\n")
      f.write("Classical ML (Prediction Probability)\n\n")
      if final_state['customer_id'] is not None:
        f.write(f"Customer_Id:\n{final_state['customer_id']}\n\n")
      if final_state['product_id'] is not None:
        f.write(f"Product_Id:\n{final_state['product_id']}\n\n")
      f.write(f"Purchase probabiity:\n{final_state['purchase_probability']}\n\n")
      f.write(f"Prediction label:\n{final_state['prediction_label']}\n\n")
      f.write(f"Agent Response:\n")
      response = final_state['final_response']
      f.write(response + "\n\n")
    elif final_state['query_type'] == 'recommendation':
      f.write("Route:\n")
      f.write("Product RAG\n\n")
      f.write(f"Agent Response:\n")
      output = final_state['final_response']
      response = output.response
      f.write(response + "\n\n")
    elif final_state['query_type'] == 'segmentation':
      f.write("Route:\n")
      f.write("Classical ML (Customer Segmentation)\n\n")
      f.write(f"Agent Response:\n")
      output = final_state['final_response']
      response = output.response
      f.write(response + "\n\n")
    else:
      f.write("Route:\n")
      f.write("Policy RAG\n\n")
      f.write(f"Agent Response:\n")
      output = final_state['final_response']
      response = output.relevant_response
      policy_id = output.policy_source[0].policy_id
      f.write(response + "\n\n")
      f.write(f"Policy ID:\n")
      f.write(policy_id + "\n\n")

  print(f"Done with the Tests execution. Please check the log files at : {log_file} and {OUTPUT_JSON_FILE}")

Done with the Tests execution. Please check the log files at : /content/capstone_project/outputs/logs/eval_run.txt and /content/capstone_project/outputs/logs/assistant_agent_evaluation.jsonl


# **Conclusion And Outcome**

In this notebook, I integrated the Product RAG, Classical ML, and Policy RAG components into a unified agentic retail assistant using LangGraph.

**Activities Performed:**

*   Loaded the trained ML pipeline. This involves the loading of Supervised Prediction model and Un-supervised Segmentation model.
*   Loaded the Product RAG and Policy RAG retrieval components.
*   Implemented a LangGraph-based orchestrator to classify and route user requests.
*   Routed requests to the appropriate specialist component:

         1) Product Rcommendatin : product recommendations and catalog-related queries.
         2) Purchase Prediction : customer/product purchase-likelihood prediction.
         3) Customer Segmentation : customer segmentation speciic queries.
         4) Policy Assistance : retail policy questions.

*   Integrated LLM-based response generation.
*   Used structured validation using Pydantic Schema where its required to maintain reliable output formats.
*   Tested the complete workflow using representative queries across all supported query types.


**Final outcome**

The individual ML, RAG, and policy components developed in the previous notebooks have been successfully integrated into a single agentic retail assistant which is capable of handling recommendation, prediction, segmentation and policy-question workflows.


